In [ ]:
from pathlib import Path
import pandas as pd

from robovast.common.campaign_data import list_run_dirs, read_test_result

DATA_DIR = ''

# Single config (DATA_DIR is its dir): aggregate its runs into one row —
# mean of per-run metrics.csv + failure_rate from test.xml.
config_dir = Path(DATA_DIR)
runs = list_run_dirs(config_dir)
rows = [pd.read_csv(r / 'metrics.csv').iloc[0]
        for r in runs if (r / 'metrics.csv').exists()]
rec = pd.DataFrame(rows).mean(numeric_only=True).to_dict() if rows else {}
failures = 0
for r in runs:
    try:
        if not read_test_result(r)['success']:
            failures += 1
    except FileNotFoundError:
        failures += 1
rec['failure_rate'] = failures / len(runs) if runs else 0.0
rec['n_runs'] = len(runs)
rec['config'] = config_dir.name

metrics = pd.DataFrame([rec])
print(f'{rec["n_runs"]} run(s) aggregated')
metrics

In [ ]:
# Single config: its aggregated metrics (one row).
metrics.T